BƯỚC 1: KHỞI TẠO, LOAD DỮ LIỆU VÀ LÀM SẠCH

In [2]:
# [BƯỚC 1] IMPORT & PRE-PROCESSING
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

# 1. Load dữ liệu
print("⏳ Đang tải dữ liệu...")
try:
    df = pd.read_csv('MBTI 500.csv') # Đảm bảo file đã upload

    # Xóa dòng lỗi/rỗng (Quan trọng)
    df.dropna(subset=['posts', 'type'], inplace=True)
    df['type'] = df['type'].astype(str)

    print(f"✅ Đã tải: {len(df)} dòng dữ liệu sạch.")
except FileNotFoundError:
    print("❌ LỖI: Không tìm thấy file 'MBTI 500.csv'.")

# 2. Hàm làm sạch (Regex)
def basic_clean(text):
    text = str(text).lower()
    # Xóa URL
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Xóa 16 từ khóa MBTI (Để tránh model học vẹt)
    mbti_types = ['infj', 'intp', 'infp', 'entp', 'intj', 'istj', 'entj', 'istp',
                  'enfj', 'enfp', 'isfj', 'esfj', 'isfp', 'estj', 'estp', 'esfp']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)
    # Chỉ giữ lại chữ và số
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

print("⏳ Đang làm sạch text (Chờ xíu)...")
df['clean_text'] = df['posts'].apply(basic_clean)
print("✅ BƯỚC 1 HOÀN TẤT: Dữ liệu đã sẵn sàng!")

⏳ Đang tải dữ liệu...
✅ Đã tải: 106067 dòng dữ liệu sạch.
⏳ Đang làm sạch text (Chờ xíu)...
✅ BƯỚC 1 HOÀN TẤT: Dữ liệu đã sẵn sàng!


BƯỚC 2: HUẤN LUYỆN (TRAINING LOOP)

In [3]:
# [BƯỚC 2] TRAINING LOOP (TF-IDF + SVM)
axes = {
    'IE': ('I', 'E'),
    'NS': ('N', 'S'),
    'TF': ('F', 'T'),
    'JP': ('J', 'P')
}

svm_results = []

print("🚀 BẮT ĐẦU CHẠY SVM...")
print("=" * 60)

for axis, (char0, char1) in axes.items():
    print(f"🔹 ĐANG XỬ LÝ TRỤC: {axis} ({char0} vs {char1})")

    # 1. Tạo nhãn (0 cho char0, 1 cho char1)
    df['label'] = df['type'].apply(lambda x: 1 if char1 in x else 0)

    # 2. Chia tập Train/Test (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
    )

    # 3. Pipeline: TF-IDF -> Linear SVM
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('clf', LinearSVC(class_weight='balanced', random_state=42, dual='auto'))
    ])

    # 4. Train
    pipeline.fit(X_train, y_train)

    # 5. Predict
    y_pred = pipeline.predict(X_test)

    # 6. Tính chỉ số
    acc = accuracy_score(y_test, y_pred)
    f1_target = f1_score(y_test, y_pred, pos_label=1)   # F1 cho nhóm thiểu số
    mf1 = f1_score(y_test, y_pred, average='macro')     # F1 trung bình (quan trọng)
    wf1 = f1_score(y_test, y_pred, average='weighted')  # F1 trọng số

    # Lưu kết quả
    svm_results.append({
        'Trục': axis,
        'Accuracy': acc,
        'F1-Target': f1_target,
        'Macro F1': mf1,
        'Weighted F1': wf1
    })

    print(f"   ✅ Xong {axis}. Acc: {acc:.4f} | Macro F1: {mf1:.4f}")

print("\n✅ BƯỚC 2 HOÀN TẤT: Đã có kết quả!")

🚀 BẮT ĐẦU CHẠY SVM...
🔹 ĐANG XỬ LÝ TRỤC: IE (I vs E)
   ✅ Xong IE. Acc: 0.8427 | Macro F1: 0.8006
🔹 ĐANG XỬ LÝ TRỤC: NS (N vs S)
   ✅ Xong NS. Acc: 0.9134 | Macro F1: 0.7789
🔹 ĐANG XỬ LÝ TRỤC: TF (F vs T)
   ✅ Xong TF. Acc: 0.8947 | Macro F1: 0.8856
🔹 ĐANG XỬ LÝ TRỤC: JP (J vs P)
   ✅ Xong JP. Acc: 0.7972 | Macro F1: 0.7933

✅ BƯỚC 2 HOÀN TẤT: Đã có kết quả!


BƯỚC 3 (Hiển thị kết quả)

In [5]:
# [BƯỚC 3] HIỂN THỊ BẢNG KẾT QUẢ CHI TIẾT (SVM)
import pandas as pd

# 1. Chuyển kết quả thành bảng DataFrame
final_df = pd.DataFrame(svm_results)

# 2. Sắp xếp thứ tự cột cho dễ nhìn
final_df = final_df[['Trục', 'Accuracy', 'F1-Target', 'Macro F1', 'Weighted F1']]

# 3. Định dạng số: Lấy 4 chữ số thập phân (Chuẩn báo cáo khoa học)
pd.options.display.float_format = '{:.4f}'.format

# 4. In ra màn hình
print("\n📊 BẢNG TỔNG HỢP KẾT QUẢ CHI TIẾT (TF-IDF + SVM)")
print("=" * 85)
print(final_df)
print("=" * 85)


📊 BẢNG TỔNG HỢP KẾT QUẢ CHI TIẾT (TF-IDF + SVM)
  Trục  Accuracy  F1-Target  Macro F1  Weighted F1
0   IE    0.8427     0.7089    0.8006       0.8484
1   NS    0.9134     0.6064    0.7789       0.9214
2   TF    0.8947     0.9179    0.8856       0.8955
3   JP    0.7972     0.8217    0.7933       0.7979
